# Build knowledge graph

In [ ]:
%env AWS_PROFILE=kmnlp

In [ ]:
import sys
sys.path.append("../src")

In [ ]:
import json
import re
from typing import Any

from pydantic import BaseModel, Field
from rich import print as rprint
import s3fs

In [ ]:
# If True, create only a single global "canonical" instance of each dim without
# the noisy long_name and units metadata fields.
# If False, create an instance for each occurence of a dim i.e. there will be
# a unique time dim for each dataset. These will also include noisy metadata
# fields.
USE_CANONICAL_DIMS = True

## Load metadata

### Load metadata for CDR datasets

In [ ]:
s3 = s3fs.S3FileSystem()

In [ ]:
with s3.open("s3://data-c6c22a2e42294c11b52ee7f0c792c071/ds_metadata.json", "r") as f:
    ds_metadata = json.load(f)

### Load metadata for CRW SST dataset

In [ ]:
crw_uri = "s3://data-c6c22a2e42294c11b52ee7f0c792c071/crw/daily_5km/zarr_reference_final.json"
with s3.open(crw_uri, "r") as f:
    crw_sst_metadata = {
        "daily_5km": {
            "uri": crw_uri,
            "zarr_metadata": {
                k: json.loads(v, strict=False) 
                for k, v in json.load(f)["refs"].items()
                if ".zattrs" in k
            }
        }
    }

In [ ]:
ds_metadata["crw"] = crw_sst_metadata

---

## Custom overrides

### Corrected values for `cdr_variable`

For Edge cases where `cdr_variable` does not match name of actual variable.

Compare `cdr_variable`, variable name in data, and `standard_name`:

In [ ]:
# for ds_name, groups in ds_metadata.items():
#     for group_name, group_data in groups.items():
#         if group_data["uri"] is None:
#             continue
#         if "cdr_variable" not in group_data["zarr_metadata"][".zattrs"]:
#             continue
#         print(ds_name, group_name)
#         print(group_data["zarr_metadata"][".zattrs"]["cdr_variable"])
#         print([
#             (k.split("/")[0], f'std_name: {v.get("standard_name")}')
#             for k, v in group_data["zarr_metadata"].items()
#             if (
#                 k.endswith("/.zattrs")
#                 and k.split("/")[0] not in {"lat", "lon", "time", "latitude", "longitude", "nv", "bounds", "crs", "wavelength"}
#                 and not any(v_ in k for v_ in {"UNC", "QA", "_bounds", "_bnds", "_qc", "_Band_Width"})
#             )
#         ])
#         print("---")

In [ ]:
custom_cdr_var_mappings = {
    "noaa-cdr-aerosol-optical-thickness-pds": {
        "daily": "aot1",
        "monthly": "aot1",
    },
    "noaa-cdr-precip-gpcp-daily-pds": {
        "daily": "precip",
    },
    "noaa-cdr-precip-gpcp-monthly-pds": {
        "monthly": "precip",
    },
    "noaa-cdr-precip-cmorph-pds": {
        "daily": "cmorph",
    },
}

### Variable names for non-CDR datasets (CRW)

In [ ]:
custom_cdr_var_mappings["crw"] = {
    "daily_5km": "analysed_sst,sea_ice_fraction",
}

### Alternate dim names

In [ ]:
dim_alt_names = {
    "latitude": "lat",
    "longitude": "lon",
}

## Parse metadata

In [ ]:
class Dimension(BaseModel):
    id: str
    standard_name: str
    long_name: str | None = None
    units: str | None = None


class Variable(BaseModel):
    id: str
    standard_name: str
    field_name: str
    long_name: str
    units: str | None = None
    dims: list[Dimension]

    
class Dataset(BaseModel):
    id: str
    title: str | None = None
    summary: str | None = None
    uri: str
    temporal_resolution: str | None
    start_date: str
    end_date: str
    variables: list[Variable]
    keywords: list[str]
    dims: list[Dimension]

In [ ]:
def parse(dataset_name: str, dataset_group: str, info: dict[str, Any]) -> Dataset:
    collection_metadata = info["zarr_metadata"][".zattrs"]
    try:
        cdr_variable = custom_cdr_var_mappings[dataset_name][dataset_group]
    except KeyError:
        cdr_variable = collection_metadata["cdr_variable"]
    var_names_in_data = [var_name.strip() for var_name in cdr_variable.split(",")]
    vars_metadata = {
        var_name: info["zarr_metadata"][f"{var_name}/.zattrs"] 
        for var_name in var_names_in_data
    }
    dim_names = list(set(d for vm in vars_metadata.values() for d in vm["_ARRAY_DIMENSIONS"]))
    dims_metadata = {
        dim_name: info["zarr_metadata"][f"{dim_name}/.zattrs"] 
        for dim_name in dim_names
    }
    dims = {
        dim_name: Dimension(
            id=f"{dim_name}_{dataset_name}_{dataset_group}", 
            **dim_metadata
        ) 
        for dim_name, dim_metadata in dims_metadata.items()
    }
    variables = []
    for var_name_in_data, v_metadata in vars_metadata.items():
        std_name = v_metadata["standard_name"].replace(" ", "_")
        v = Variable(
            id=f"{std_name}_{dataset_name}_{dataset_group}",
            standard_name=std_name,
            field_name=var_name_in_data,
            long_name=v_metadata["long_name"],
            units=v_metadata["units"],
            dims=[dims[dim_name] for dim_name in v_metadata["_ARRAY_DIMENSIONS"]],
        )
        variables.append(v)
    ds = Dataset(
        id=f"{dataset_name}_{dataset_group}",
        title=collection_metadata.get("title"),
        summary=collection_metadata.get("summary"),
        uri=info["uri"],
        temporal_resolution=collection_metadata.get("time_coverage_resolution"),
        start_date=collection_metadata.get("time_coverage_start"),
        end_date=collection_metadata.get("time_coverage_end"),
        variables=variables,
        keywords=set(re.split(r"\s*,\s*", collection_metadata.get("keywords", ""))),
        dims=list(dims.values()),
    )
    return ds

In [ ]:
# rprint(
#     parse(
#         "noaa-cdr-solar-spectral-irradiance-pds",
#         "yearly",
#         ds_metadata["noaa-cdr-solar-spectral-irradiance-pds"]["yearly"],
#     )
# )

In [ ]:
datasets = []
for ds_name, groups in ds_metadata.items():
    for group_name, group_data in groups.items():
        if group_data["uri"] is None:
            continue
        try:
            dataset = parse(ds_name, group_name, group_data)
            datasets.append(dataset)
        except KeyError:
            print(ds_name, group_name)
            continue


---

## Populate graph

In [ ]:
from rdflib import Graph, Literal, Namespace, BNode, RDF, RDFS, URIRef, DCTERMS
from rdflib.namespace import XSD, OWL, DCAT, TIME, SDO, QB

In [ ]:
# Define Namespaces
BASE = Namespace("http://kmnlp-demo.noaa.gov/")
QUDT = Namespace("http://qudt.org/3.1.1/schema/qudt/")
UNIT = Namespace("http://qudt.org/3.1.1/vocab/unit/")
CF = Namespace("http://mmisw.org/ont/cf/parameter/")

In [ ]:
def add_dataset(g: Graph, dataset: Dataset, canonical_dims: bool = False) -> str:
    """Add a dataset to the graph."""

    # add dims
    if not canonical_dims:
        for d in dataset.dims:
            add_dim(g, d)
    
    # add vars
    for v in dataset.variables:
        add_variable(g, v)

    ds_uri = BASE[dataset.id]

    # rdf
    g.add((ds_uri, RDF.type, DCAT.Dataset))
    g.add((ds_uri, RDF.type, SDO.Dataset))
    g.add((ds_uri, RDF.type, QB.DataSet))

    # schema.org properties
    g.add((ds_uri, SDO.version, Literal("2")))
    if dataset.title:
        g.add((ds_uri, SDO.name, Literal(dataset.title)))
    if dataset.summary:
        g.add((ds_uri, SDO.description, Literal(dataset.summary)))
    g.add((ds_uri, SDO.contentUrl, URIRef(dataset.uri)))
    g.add((ds_uri, SDO.isAccessibleForFree, Literal(True)))
    # g.add((ds_uri, SDO.identifier, Literal("")))
    for k in dataset.keywords:
        g.add((ds_uri, SDO.keywords, Literal(k)))
    for dim in dataset.dims:
        dim_iri = BASE[dim.standard_name if canonical_dims else dim.id]
        g.add((ds_uri, SDO.variableMeasured, dim_iri))
    for v in dataset.variables:
        g.add((ds_uri, SDO.variableMeasured, BASE[v.id]))

    # dcterms
    if dataset.title:
        g.add((ds_uri, DCTERMS.title, Literal(dataset.title)))
    if dataset.summary:
        g.add((ds_uri, DCTERMS.description, Literal(dataset.summary)))

    # qb
    for v in dataset.variables:
        v_structure_def_bnode = BNode()
        g.add((v_structure_def_bnode, RDF.type, QB.DataStructureDefinition))
        for dim in v.dims:
            qb_dim_bnode = BNode()
            dim_iri = BASE[dim.standard_name if canonical_dims else dim.id]
            g.add((qb_dim_bnode, QB.dimension, dim_iri))
            g.add((v_structure_def_bnode, QB.component, qb_dim_bnode))
        qb_var_bnode = BNode()
        g.add((qb_var_bnode, QB.measure, BASE[v.id]))
        g.add((v_structure_def_bnode, QB.component, qb_var_bnode))
        g.add((ds_uri, QB.structure, v_structure_def_bnode))

    # rdfs
    if dataset.title:
        g.add((ds_uri, RDFS.label, Literal(dataset.title)))
    if dataset.summary:
        g.add((ds_uri, RDFS.comment, Literal(dataset.summary)))

    # dcat
    dcat_dist_bnode = BNode()
    g.add((dcat_dist_bnode, RDF.type, DCAT.Distribution))
    g.add((dcat_dist_bnode, DCAT.accessURL, URIRef(dataset.uri)))
    g.add((dcat_dist_bnode, DCAT.mediaType, Literal("application/zarr")))
    g.add((ds_uri, DCAT.distribution, dcat_dist_bnode))
    if dataset.temporal_resolution:
        g.add((ds_uri, DCAT.temporalResolution, Literal(dataset.temporal_resolution, datatype=XSD.duration)))
    g.add((ds_uri, DCAT.startDate, Literal(dataset.start_date, datatype=XSD.dateTime)))
    g.add((ds_uri, DCAT.endDate, Literal(dataset.end_date, datatype=XSD.dateTime)))

    return ds_uri


def add_variable(g: Graph, variable: Variable) -> str:
    """Add a variable to the graph."""
    var_uri = BASE[variable.id]

    # rdf
    g.add((var_uri, RDF.type, SDO.PropertyValue))
    g.add((var_uri, RDF.type, QB.MeasureProperty))

    # schema.org properties
    g.add((var_uri, SDO.name, Literal(variable.standard_name)))
    g.add((var_uri, SDO.description, Literal(variable.long_name)))
    g.add((var_uri, SDO.alternateName, Literal(variable.field_name)))
    if variable.units:
        g.add((var_uri, SDO.unitText, Literal(variable.units)))
    g.add((var_uri, SDO.propertyID, QB.measure))

    # rdfs
    g.add((var_uri, RDFS.label, Literal(variable.long_name)))

    # qudt
    # g.add((var_uri, QUDT.hasUnit, UNIT[""]))

    # cf
    g.add((var_uri, BASE.cf_standard_name, CF[variable.standard_name]))

    return var_uri


def add_dim(g: Graph, dim: Dimension) -> str:
    """Add a dimension to the graph."""
    std_name = dim.standard_name
    dim_uri = BASE[dim.id]

    # rdf
    g.add((dim_uri, RDF.type, SDO.PropertyValue))
    g.add((dim_uri, RDF.type, QB.DimensionProperty))

    # schema.org
    if std_name in dim_alt_names:
        g.add((dim_uri, SDO.alternateName, Literal(dim_alt_names[std_name])))
    if dim.units:
        g.add((dim_uri, SDO.unitText, Literal(dim.units)))
    g.add((dim_uri, SDO.propertyID, QB.measureDimension))

    # rdfs
    g.add((dim_uri, RDFS.label, Literal(dim.long_name)))

    # cf
    g.add((dim_uri, BASE.cf_standard_name, CF[std_name]))

    return dim_uri


def add_dim_canonical(g: Graph, dim: Dimension) -> str:
    """Add a canonical dimension to the graph."""
    std_name = dim.standard_name
    dim_uri = BASE[std_name]

    # rdf
    g.add((dim_uri, RDF.type, SDO.PropertyValue))
    g.add((dim_uri, RDF.type, QB.DimensionProperty))

    # schema.org
    g.add((dim_uri, SDO.name, Literal(std_name)))
    if std_name in dim_alt_names:
        g.add((dim_uri, SDO.alternateName, Literal(dim_alt_names[std_name])))
    g.add((dim_uri, SDO.propertyID, QB.measureDimension))

    # rdfs
    g.add((dim_uri, RDFS.label, Literal(std_name)))

    # cf
    g.add((dim_uri, BASE.cf_standard_name, CF[std_name]))

    return dim_uri

In [ ]:
g = Graph()

# bind prefixes
g.bind("", BASE)
g.bind("cf", CF)
g.bind("dcat", DCAT)
g.bind("dcterms", DCTERMS)
g.bind("owl", OWL)
g.bind("qb", QB)
g.bind("qudt", QUDT)
g.bind("rdf", RDF)
g.bind("rdfs", RDFS)
g.bind("schema", SDO)
g.bind("time", TIME)
g.bind("unit", UNIT)
g.bind("xsd", XSD)

# custom properties
cf_standard_name = BASE["cf_standard_name"]
g.add((cf_standard_name, RDF.type, RDF.Property))
g.add((cf_standard_name, RDFS.label, Literal("Matching entry in the CF Standard Names table.")))
g.add((cf_standard_name, RDFS.range, CF.Standard_Name))

if USE_CANONICAL_DIMS:
    dims_added = set()
    for ds in datasets:
        for dim in ds.dims:
            if dim.standard_name in dims_added:
                continue
            add_dim_canonical(g, dim)
            dims_added.add(dim.standard_name)

for ds in datasets:
    add_dataset(g, ds, canonical_dims=USE_CANONICAL_DIMS)

## Save

In [ ]:
ttl_str = g.serialize(format="turtle")

In [ ]:
out_path = "../src/llm_agent/data/dataset-kg-generated.ttl"
with open(out_path, "w") as f:
    f.write(ttl_str.rstrip())
    f.write("\n")